In [93]:
import pandas as pd 
import ast
import numpy as np
movies=pd.read_csv("tmdb_5000_movies.csv")
credits=pd.read_csv("tmdb_5000_credits.csv")

movies=movies.merge(credits,on="title")

print(movies.shape)
print(movies.head())

(4809, 23)
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   
2  [{"id"

In [94]:
movies["vote_average"].min(), movies["vote_average"].max()

(np.float64(0.0), np.float64(10.0))

In [95]:
movies["cast"].iloc[0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [96]:
movies.isnull().sum()

budget                     0
genres                     0
homepage                3096
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
movie_id                   0
cast                       0
crew                       0
dtype: int64

In [97]:
print(movies.columns)

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')


In [98]:
movies=movies[[ "budget", "genres","keywords","overview","popularity","release_date","runtime","title","vote_average","cast","crew"]]

print(movies.shape)

movies.dropna(subset=["cast","crew","title", "overview"], inplace=True)

movies["vote_average"].fillna(movies["vote_average"].median())

movies["budget"].replace(0, movies["budget"].median())

movies["popularity"].fillna(movies["popularity"].median())

movies["year"] = movies["release_date"].apply(lambda x: int(x.split("-")[0]) if pd.notnull(x) else None)



movies.drop(columns=["release_date"], inplace=True)


(4809, 11)


In [99]:
movies.dropna(subset=["year"], axis=0, inplace=True)

In [100]:
movies.isnull().sum()


budget          0
genres          0
keywords        0
overview        0
popularity      0
runtime         0
title           0
vote_average    0
cast            0
crew            0
year            0
dtype: int64

In [101]:
from rapidfuzz import process, fuzz


In [102]:
def filter_movies(df, filters):

    if filters["rating"] is not None:
        df = df[df["vote_average"] >= filters["rating"]]

    if filters["year"] is not None:
        df = df[df["year"] == filters["year"]]

    if filters.get("year_after") is not None:
        df = df[df["year"] >= filters["year_after"]]

    if filters.get("year_before") is not None:
        df = df[df["year"] < filters["year_before"]]


# #adding fuzzy (actor)
#     if filters["actor"] is not None:
#         actor_name = filters["actor"]

#         def match_actor(x):
#             if x is None:
#                 return False

#             if isinstance(x, str):
#                 names = [i.strip().lower() for i in x.split(",")]
#             else:
#                 names = [str(i).lower() for i in x]

#             for name in names:
#                 if actor_name == name:
#                     return True

#                 score = fuzz.token_set_ratio(actor_name, name)

#                 if len(actor_name) <= 4:
#                     if score > 95:
#                         return True
#                 else:
#                     if score > 85:
#                         return True

#             return False

#         df = df[df["cast"].apply(match_actor)]
    

# #adding fuzzy (director)
#     if filters["director"] is not None:
#         director_name = filters["director"].lower()

#         def match_director(x):
#             if x is None :
#                 return False

        
#             if isinstance(x, str):
#                 names = [i.strip().lower() for i in x.split(",")]
#             else:
#                 names = [str(i).lower() for i in x]

#             for name in names:
#                 score = fuzz.token_set_ratio(director_name, name)

#                 if len(director_name) <= 4:
#                     if score > 95:
#                         return True
#                 else:
#                     if score > 85:
#                         return True

#             return False

#         df = df[df["crew"].apply(match_director)]


#     df = df.sort_values(by=["vote_average","popularity"], ascending=False)

#     return df
#     if filters["actor"] is not None:
#         actor_name = filters["actor"]

#         def match_actor(x):
#             for i in x:
#                 i_clean=i.lower()

#                 if actor_name==i_clean:
#                     return True 
#                 score = fuzz.token_set_ratio(actor_name, i)

#             # ✅ short query (like "tom")
#                 if len(actor_name) <= 4:
#                     if score > 95:
#                         return True

#             # ✅ normal query ("brad pitt")
#                 else:
#                     if score > 85:
#                         return True

#             return False
#         # df = df[df["cast"].apply(lambda x: any(fuzz.token_set_ratio(actor_name, i) > 90 and abs(len(actor_name) - len(i)) < 5 for i in x))]
#         df = df[df["cast"].apply(match_actor)]
    

#     # if filters["director"] is not None:
#     #     director_name = filters["director"].replace(" ", "").lower()
#     #     df = df[df["crew"].apply(lambda x: any(director_name == i.replace(" ", "").lower() for i in x))]
# #adding fuzzy (director)
#     if filters["director"] is not None:
#         director_name = filters["director"]

#         def match_director(x):
#             for i in x:
#                 score = fuzz.token_set_ratio(director_name, i)

#             # ✅ short query (like "tom")
#                 if len(director_name) <= 4:
#                     if score > 95:
#                         return True

#             # ✅ normal query ("christopher nolan")
#                 else:
#                     if score > 85:
#                         return True

#             return False
#         # df = df[df["crew"].apply(lambda x: any(fuzz.token_set_ratio(director_name, i) > 90 for i in x))]
#         df = df[df["crew"].apply(match_director)]

#     df = df.sort_values(by=["vote_average","popularity"], ascending=False)

#     return df


# trial 1 

    if filters["actor"] is not None:
        actor_name = filters["actor"].lower().strip()

        def match_actor(x):
            if not isinstance(x, list):
                return False

            for name in x:
                name = name.lower().strip()

            # exact match
                if actor_name == name:
                    return True

                score = fuzz.partial_ratio(actor_name, name)

                if len(actor_name) <= 4:
                    if score > 90:   # reduced threshold
                        return True
                else:
                    if score > 80:
                        return True

            return False

        df = df[df["cast"].apply(match_actor)]

    if filters["director"] is not None:
        director_name = filters["director"].lower().strip()

        def match_director(x):
            if not isinstance(x, list):
                return False

            for name in x:
                name = name.lower().strip()

                score = fuzz.partial_ratio(director_name, name)

                if len(director_name) <= 4:
                    if score > 90:
                        return True
                else:
                    if score > 80:
                        return True

            return False

        df = df[df["crew"].apply(match_director)]

        df= df.sort_values(by=["vote_average","popularity"], ascending=False)

    return df

In [103]:
filters = {"rating": 7,"year": 2010,"actor": None,"director": None}
result = filter_movies(movies, filters)
print(result["title"].head(10))

6                           Tangled
42                      Toy Story 3
92         How to Train Your Dragon
96                        Inception
411     Scott Pilgrim vs. the World
439                  Shutter Island
614                   Despicable Me
1162             The Social Network
1329                       The Town
1367                      True Grit
Name: title, dtype: object


In [104]:
filters = {"rating": None, "year": None, "actor": "Tom", "director": None}
result = filter_movies(movies, filters)
print(result["title"].head(10))

Series([], Name: title, dtype: object)


In [105]:
print(movies["genres"].iloc[0])
print(type(movies["genres"].iloc[0]))

[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]
<class 'str'>


In [106]:
import ast

# def convert(text):
#     return [i['name'] for i in ast.literal_eval(text)]
def convert(text):
    try:
        data = ast.literal_eval(text)
        
        # case 1: list of dicts
        if isinstance(data, list) and len(data) > 0 and isinstance(data[0], dict):
            return [i['name'] for i in data]
        
        # case 2: already list of strings
        elif isinstance(data, list):
            return data
        
    except:
        pass
    
    return []

# def get_director(text):
#     return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']
def get_director(text):
    try:
        # Case 1: already a list
        if isinstance(text, list):
            data = text
        else:
            data = ast.literal_eval(text)

        return [
            i['name']
            for i in data
            if isinstance(i, dict) and i.get('job') == 'Director'
        ]

    except Exception as e:
        return []

for col in ["genres", "keywords", "cast"]:
    movies[col] = movies[col].apply(convert)

movies["crew"] = movies["crew"].apply(get_director)

In [107]:
movies['combined'] = (
    movies['overview']+ " " + movies['overview']+ " " +
    movies['genres'].apply(lambda x: " ".join(x)) + " " +
    movies['keywords'].apply(lambda x: " ".join(x)) + " " +
    movies['cast'].apply(lambda x: " ".join(x[:3])) #+ " " +
    #movies['crew'].apply(lambda x: " ".join(x))
)

print(movies["combined"].iloc[0])

movies.to_csv("cleaned_movies.csv", index=False)

In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy Science Fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d Sam Worthington Zoe Saldana Sigourney Weaver


In [108]:
!pip show sentence-transformers


Name: sentence-transformers
Version: 5.3.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: C:\Users\LP\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 


In [126]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

movie_embeddings = model.encode(movies['combined'].tolist(),show_progress_bar=True)


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query, top_k=5):
    
    query=query+"movie plot story theme emotion character journey drama"
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, movie_embeddings)[0]
    top_k=min(top_k, len(similarities))
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return movies.iloc[top_indices]


def semantic_search_filtered(df,query, top_k=5):

    df=df.reset_index(drop=True)
    query=query+"movie plot story theme emotion character journey drama"
    query_embedding = model.encode([query])
    subset_embeddings=movie_embeddings[df.index]
    similarities = cosine_similarity(query_embedding, subset_embeddings)[0]
    top_k=min(top_k, len(similarities))
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    return df.iloc[top_indices]



In [111]:
print(semantic_search("feel good movies")["title"])
print(semantic_search("hero is an outcast")["title"])
print(semantic_search("rags to riches story")["title"])
query = "uplifting happy emotional inspiring movie"
print(semantic_search(query)["title"])

2684               Poetic Justice
1334                     Magnolia
3873    To Write Love on Her Arms
2895            Darling Companion
4052              Redemption Road
Name: title, dtype: object
2650               Winnie Mandela
4052              Redemption Road
1242               Sweet November
2684               Poetic Justice
3873    To Write Love on Her Arms
Name: title, dtype: object
2650        Winnie Mandela
1334              Magnolia
1558          Mystic River
3520    Trade Of Innocents
3112                Edmond
Name: title, dtype: object
2895            Darling Companion
1334                     Magnolia
3873    To Write Love on Her Arms
4217             I Love Your Work
4052              Redemption Road
Name: title, dtype: object


In [112]:
def hybrid_search(movies, filters, query=None, top_k=5):
    
    filtered_df = filter_movies(movies, filters)

    if filtered_df.empty:
        

        relaxed_df = filter_movies(movies, {
            "rating": filters.get("rating"),
            "year": None,
            "year_after": None,
            "year_before": None,
            "actor": filters.get("actor"),
            "director": filters.get("director")
        })

        
        if not relaxed_df.empty:
            return relaxed_df.head(min(top_k, len(relaxed_df)))
        
        print("No exact match, relaxing filters...")

        if query:
            return semantic_search(query, top_k)
        
        return movies.head(top_k)

    # normal flow
    if query:
        return semantic_search_filtered(filtered_df, query, top_k)

    return filtered_df.head(min(top_k, len(filtered_df)))

In [113]:
filters = {"rating": None,"year": None,"actor": "Brad Pitt","director": None}

print(hybrid_search(movies, filters)["title"])

filters = {"rating": None, "year": None, "actor": "Tom Cruise", "director": "Christopher Nolan"}
print(hybrid_search(movies, filters, "feel good movies")["title"])

45                             World War Z
100    The Curious Case of Benjamin Button
145                                   Troy
173                         Happy Feet Two
196                               Megamind
Name: title, dtype: object
0    Mission: Impossible - Rogue Nation
1                            Young Guns
2                          Jack Reacher
Name: title, dtype: object


In [114]:
import re

def extract_top_k(query):
    match = re.search(r'(\d+)\s+(movies|films)', query.lower())
    if match:
        return int(match.group(1))
    return 5


In [115]:
!pip show huggingface_hub
!pip show transformers

Name: huggingface-hub
Version: 0.35.3
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache
Location: C:\Users\LP\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: filelock, fsspec, packaging, pyyaml, requests, tqdm, typing-extensions
Required-by: accelerate, faster-whisper, sentence-transformers, tokenizers, transformers
Name: transformers
Version: 4.57.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: C:\Users\LP\AppData\Local\Programs\Python\Python312\Lib\site-packages
R

In [116]:
%pip install transformers accelerate torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [117]:
# from transformers import pipeline

import requests
import os
from dotenv import load_dotenv
load_dotenv()
API_URL = "https://api-inference.huggingface.co/models/google/flan-t5-small"
HEADERS = {
    "Authorization": "Bearer " + os.getenv("HF_TOKEN") 
}

def query_llm(prompt):
    try:
        response = requests.post(
            API_URL,
            headers=HEADERS,
            json={
                "inputs": prompt,
                "parameters": {"max_new_tokens": 100}
            }
        )

        result = response.json()

        if isinstance(result, list):
            return result[0]["generated_text"]

        return ""

    except:
        return ""

# llm=pipeline(
#     "text-generation",
#     model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     device_map="auto"
# )



# response = llm("Hello", max_new_tokens=50)
# print(response[0]["generated_text"])

In [118]:
def extract_filters(query):
    
    filters = {
        "rating": None,
        "year": None,
        "year_after": None,
        "year_before": None,
        "actor": None,
        "director": None
    }
    
    query = query.lower()
    
    # rating
    rating_match = re.search(r'rating (above|greater than)?\s*(\d+(\.\d+)?)', query)
    if rating_match:
        filters["rating"] = float(rating_match.group(2))
    
    # year
    # year exact
    year_match = re.search(r'(19|20)\d{2}', query)
    if year_match:
        filters["year"] = int(year_match.group())

    # year after
    after_match = re.search(r'after\s+(19|20)\d{2}', query)
    if after_match:
        filters["year_after"] = int(after_match.group().split()[-1])

    # year before
    before_match = re.search(r'before\s+(19|20)\d{2}', query)
    if before_match:
        filters["year_before"] = int(before_match.group().split()[-1])
    
    # actor
    actor_match = re.search(r'(acted by|starring|with)\s+([a-zA-Z\s]+?)(?:\sin|\swith|\sdirected|$)', query)
    if actor_match:
        filters["actor"] = actor_match.group(2).strip()
    
    # director
    director_match = re.search(r'(directed by)\s+([a-zA-Z\s]+)', query)
    if director_match:
        filters["director"] = director_match.group(2).strip()
    
    return filters

import json

def extract_filters_llm(query):

    prompt = f"""
Extract movie filters from this query and return ONLY JSON:

Query: "{query}"

Format:
{{
    "rating": null,
    "year": null,
    "year_after": null,
    "year_before": null,
    "actor": null,
    "director": null
}}
"""

    response = query_llm(prompt)

    try:
        start = response.find("{")
        end = response.rfind("}") + 1
        filters = json.loads(response[start:end])
        return filters

    except:
        return extract_filters(query)

In [119]:
query = "movies with rating above 7 in 2010 acted by Tom"
print(extract_filters(query))

{'rating': 7.0, 'year': 2010, 'year_after': None, 'year_before': None, 'actor': 'tom', 'director': None}


In [120]:

domain_text = "movie film actor director cinema story plot"
domain_embedding = model.encode([domain_text])
last_results = None

def is_movie_query(query):

    query_embedding = model.encode([query])

    score = cosine_similarity(query_embedding, domain_embedding)[0][0]

    return score > 0.2 

def is_new_query(query):
    
    query = query.lower()
    
    new_query_keywords = [
        "movies", "movie", "films",
        "actor", "acted", "director",
        "rating", "genre"
    ]
    
    return any(word in query for word in new_query_keywords)

# def is_followup_query(query):
    
#     query = query.lower()

#     followup_words = [
#         "which", "that", "those", "these",
#         "released", "year", "rating", "only",
#         "after", "before"
#     ]

#     return any(word in query for word in followup_words)


In [121]:
memory = {
    "rating": None,
    "year": None,
    "year_after": None,
    "year_before": None,
    "actor": None,
    "director": None
}

def update_memory(old_filters, new_filters):

    if new_filters.get("actor") is not None:
        old_filters["actor"] = new_filters["actor"]
        old_filters["director"] = None

    
    elif new_filters.get("director") is not None:
        old_filters["director"] = new_filters["director"]
        old_filters["actor"] = None

    
    for key in ["rating", "year", "year_after", "year_before"]:
        if new_filters.get(key) is not None:
            old_filters[key] = new_filters[key]
    
    return old_filters

def reset_memory():
    global memory
    memory = {key: None for key in memory}

def is_followup_query(query):

    if not any(memory.values()):
        return False  

    query_embedding = model.encode([query])

    
    memory_text = " ".join([str(v) for v in memory.values() if v])

    if not memory_text:
        return False

    memory_embedding = model.encode([memory_text])

    score = cosine_similarity(query_embedding, memory_embedding)[0][0]

    return score > 0.25

In [122]:
def chatbot(query):

    global memory
    global last_results

    if "explain" in query:
        if last_results is not None and len(last_results) > 0:

            if "first" in query:
                idx = 0
            elif "second" in query:
                idx = 1
            elif "third" in query:
                idx = 2
            else:
                idx = 0  # default

            if idx < len(last_results):
                movie = last_results.iloc[idx]

                if "overview" in movie:
                    return f"{movie['title']}:\n{movie['overview']}"
                else:
                    return f"{movie['title']}:\nNo description available."

        return "I don’t have a previous movie list to explain."


    if not is_movie_query(query) and not is_followup_query(query):
        return ["Sorry, I can only help with movie-related queries."]

    if is_new_query(query):
        memory = {key: None for key in memory}

    filters = extract_filters_llm(query)
    if not any(filters.values()):
        filters = extract_filters(query)
    if filters["actor"]:
        filters["actor"] = filters["actor"].strip().lower()

    if filters["director"]:
        filters["director"] = filters["director"].strip().lower()
    memory=update_memory(memory, filters)
    top_k= extract_top_k(query)

    if any([filters["rating"], filters["year"], filters["actor"], filters["director"]]):
        results = hybrid_search(movies, memory, None, top_k)
    else:
        results = hybrid_search(movies, memory, query, top_k)
    if isinstance(results, list):
        return results
    
    if isinstance(results,str):
        return results
    # if "title" in results:
    #     return results["title"].tolist()

    if "title" in results:
        titles = results["title"].tolist()

    if not titles:
        return ["No movies found for your query."]

    # dynamic message
    if filters["actor"]:
        intro = f"Here are movies featuring {filters['actor'].title()}:"
    elif filters["director"]:
        intro = f"Here are movies directed by {filters['director'].title()}:"
    elif filters["rating"] or filters["year"]:
        intro = "Here are movies matching your filters:"
    else:
        intro = "Here are some movies you might like:"

    formatted = [intro]

    for i, movie in enumerate(titles, 1):
        formatted.append(f"{i}. {movie}")
    last_results = results   

    return "\n".join(formatted)
    


In [123]:
print(chatbot("movies with rating above 7 in 2010"))
print(chatbot("feel good movies"))
print(chatbot("3 movies acted by brad pitt"))
print(chatbot("movies acted by Brad Pitt"))
print(chatbot("movies directed by Christopher Nolan"))
print(chatbot("movies with rating 8.0 released in 2010 directed by Christopher Nolan"))
print(chatbot("movies with rating 8 and acted by Tom Cruise"))
print(chatbot("movies where hero is an outcast"))
print(chatbot("movies acted by Tom cruise"))
print(chatbot("hero is born rich" ))
print(chatbot("hero is born poor then becomes rich"))
print(chatbot("which sport does messi play "))

Here are movies matching your filters:
1. Tangled
2. Toy Story 3
3. How to Train Your Dragon
4. Inception
5. Scott Pilgrim vs. the World
Here are some movies you might like:
1. Poetic Justice
2. Magnolia
3. To Write Love on Her Arms
4. Darling Companion
5. Redemption Road
Here are movies featuring Brad Pitt:
1. World War Z
2. The Curious Case of Benjamin Button
3. Troy
Here are movies featuring Brad Pitt:
1. World War Z
2. The Curious Case of Benjamin Button
3. Troy
4. Happy Feet Two
5. Megamind
Here are movies directed by Christopher Nolan:
1. The Dark Knight
2. Interstellar
3. Inception
4. Memento
5. The Prestige
Here are movies directed by Christopher Nolan:
1. Inception
No exact match, relaxing filters...
Here are movies featuring Tom Cruise:
1. Avatar
2. Pirates of the Caribbean: At World's End
3. Spectre
4. The Dark Knight Rises
5. John Carter
Here are some movies you might like:
1. Your Highness
2. Poetic Justice
3. Trees Lounge
4. Darling Companion
5. To Write Love on Her Arms


In [124]:
print(chatbot("movies with rating above 7"))
print(chatbot("which were released in 2010"))

Here are movies matching your filters:
1. Avatar
2. The Dark Knight Rises
3. Tangled
4. Avengers: Age of Ultron
5. Harry Potter and the Half-Blood Prince
Here are movies matching your filters:
1. Tangled
2. Toy Story 3
3. How to Train Your Dragon
4. Inception
5. Scott Pilgrim vs. the World


In [125]:
print(chatbot("movies acted by tom cruise"))
print(chatbot("hero dies in the end"))

Here are movies featuring Tom Cruise:
1. Edge of Tomorrow
2. Mission: Impossible - Rogue Nation
3. Mission: Impossible III
4. Mission: Impossible - Ghost Protocol
5. The Last Samurai
Here are some movies you might like:
1. Mission: Impossible - Rogue Nation
2. The Last Samurai
3. Knight and Day
4. Losin' It
5. Minority Report
